# Training a Holmesian Reasoning Model
## From narrative corpus to evidence-grounded investigation

**A realistic Google Colab experiment using QLoRA**

This notebook fine-tunes a small pretrained instruction model to follow a
Holmesian investigation protocol:

1. separate observations from assumptions;
2. generate competing hypotheses;
3. derive predictions;
4. identify contradictions;
5. choose a discriminating next test;
6. conclude with calibrated confidence.

It is a teaching and research prototype—not a claim that a small Colab model
has acquired human-level reasoning.

## What makes this a reasoning experiment?

The first Holmes notebook trained a causal Transformer on prose. That can learn
style and local continuation. Here we train on **structured investigation
episodes**. Each conclusion must cite evidence IDs, and a verifier rejects
nonexistent evidence references.

The training corpus has three layers:

| Layer | Purpose |
|---|---|
| Public-domain Holmes canon | provenance and qualitative inspection |
| Six curated case reconstructions | connect the curriculum to Holmes stories |
| Procedural investigation cases | create enough varied supervision for QLoRA |

Two complete procedural case families are excluded from training and used only
for out-of-distribution evaluation.

## Colab runtime and expected cost

Choose **Runtime → Change runtime type → T4 GPU**, then run all cells.

- Default model: `Qwen/Qwen2.5-0.5B-Instruct`
- Method: 4-bit NF4 quantization + LoRA
- Default sequence length: 640 tokens
- Standard run: roughly 500 curriculum examples and about 120 optimizer steps
- Typical T4 runtime: approximately 20–45 minutes, varying with Colab load
- No paid API is required

Use `RUN_MODE = "smoke"` first if you want a short end-to-end systems test.

In [ ]:
!pip -q install -U \
  "transformers>=4.46,<5" \
  "peft>=0.13,<1" \
  "accelerate>=1.1,<2" \
  "bitsandbytes>=0.44,<1" \
  "datasets>=3.1,<4" \
  "sentencepiece>=0.2,<1" \
  "scikit-learn>=1.4,<2" \
  "matplotlib>=3.8,<4"

In [ ]:
import gc
import hashlib
import inspect
import io
import json
import math
import os
import random
import re
import time
import urllib.request
import zipfile
from collections import defaultdict
from copy import deepcopy
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    get_cosine_schedule_with_warmup,
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# Choose: "smoke", "standard", or "extended".
RUN_MODE = "standard"

PRESETS = {
    "smoke": {
        "cases_per_family": 4,
        "epochs": 1,
        "max_optimizer_steps": 12,
        "max_length": 512,
        "eval_cases": 3,
    },
    "standard": {
        "cases_per_family": 30,
        "epochs": 2,
        "max_optimizer_steps": 160,
        "max_length": 640,
        "eval_cases": 10,
    },
    "extended": {
        "cases_per_family": 100,
        "epochs": 2,
        "max_optimizer_steps": 600,
        "max_length": 768,
        "eval_cases": 24,
    },
}

CFG = PRESETS[RUN_MODE]
BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
# For a stronger but slower T4 experiment, try:
# BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

OUTPUT_DIR = Path("/content/holmesian_reasoning_adapter")
MAX_LENGTH = CFG["max_length"]
MICRO_BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 8
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 0.01
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

assert RUN_MODE in PRESETS
if not torch.cuda.is_available():
    raise RuntimeError(
        "A CUDA GPU is required for the default QLoRA run. "
        "In Colab choose Runtime > Change runtime type > T4 GPU."
    )

major, minor = torch.cuda.get_device_capability()
COMPUTE_DTYPE = torch.bfloat16 if major >= 8 else torch.float16
print("Mode:", RUN_MODE)
print("Compute dtype:", COMPUTE_DTYPE)
print("Configuration:", json.dumps(CFG, indent=2))

## 1. Canon provenance

The raw stories are useful for studying language and reconstructing cases, but
they are **not** fed directly into the supervised reasoning objective. A story
usually reveals its solution later, so naïve chunking creates solution leakage.

This cell downloads selected public-domain texts and records their hashes. If a
download fails, training can still proceed because the curated reconstructions
and procedural curriculum are embedded below.

In [ ]:
GUTENBERG_BOOKS = {
    "The Adventures of Sherlock Holmes": 1661,
    "The Memoirs of Sherlock Holmes": 834,
    "The Return of Sherlock Holmes": 108,
    "His Last Bow": 2350,
    "A Study in Scarlet": 244,
    "The Sign of the Four": 2097,
}

def download_gutenberg(book_id, timeout=30):
    urls = [
        f"https://www.gutenberg.org/cache/epub/{book_id}/pg{book_id}.txt",
        f"https://www.gutenberg.org/files/{book_id}/{book_id}-0.txt",
        f"https://www.gutenberg.org/files/{book_id}/{book_id}.txt",
    ]
    last_error = None
    for url in urls:
        try:
            req = urllib.request.Request(
                url, headers={"User-Agent": "HolmesianReasoningResearch/1.0"}
            )
            with urllib.request.urlopen(req, timeout=timeout) as response:
                raw = response.read()
            return url, raw.decode("utf-8", errors="replace")
        except Exception as exc:
            last_error = exc
    raise RuntimeError(f"Could not download Gutenberg book {book_id}: {last_error}")

canon = {}
provenance = []
for title, book_id in GUTENBERG_BOOKS.items():
    try:
        url, text = download_gutenberg(book_id)
        canon[title] = text
        provenance.append({
            "title": title,
            "gutenberg_id": book_id,
            "url": url,
            "characters": len(text),
            "sha256": hashlib.sha256(text.encode("utf-8")).hexdigest(),
            "status": "downloaded",
        })
    except Exception as exc:
        provenance.append({
            "title": title,
            "gutenberg_id": book_id,
            "status": f"skipped: {exc}",
        })

print(json.dumps(provenance, indent=2)[:5000])
print(f"\nDownloaded {len(canon)} of {len(GUTENBERG_BOOKS)} books.")

## 2. The machine-readable reasoning contract

Every full answer uses the same contract. Evidence is identified as `E1`,
`E2`, and so forth. Hypotheses must cite those IDs; the conclusion must do the
same. This does not guarantee truth, but it makes unsupported references
detectable.

In [ ]:
SYSTEM_PROMPT = (
    "You are an evidence-grounded investigative reasoner. "
    "Use only the evidence supplied in the case. Separate observed facts from "
    "assumptions. Consider at least two competing hypotheses. Cite evidence only "
    "by its supplied ID. Never invent an observation. Prefer a discriminating "
    "next test over a dramatic guess. Return valid JSON only, with no markdown "
    "fences."
)

REQUIRED_FULL_KEYS = {
    "facts", "assumptions", "hypotheses",
    "best_next_test", "conclusion", "unknowns"
}

TASK_INSTRUCTIONS = {
    "OBSERVATION_AUDIT": (
        "Return JSON with keys facts and assumptions. Each fact must contain "
        "evidence_id and statement. Assumptions are unverified claims."
    ),
    "HYPOTHESIS_TEST": (
        "Return JSON with keys hypotheses and best_next_test. Supply at least "
        "two hypotheses. Each hypothesis needs id, claim, support, "
        "contradictions, and prediction."
    ),
    "FULL_ANALYSIS": (
        "Return JSON with keys facts, assumptions, hypotheses, best_next_test, "
        "conclusion, and unknowns. The conclusion needs claim, support, and "
        "confidence between 0 and 1."
    ),
}

def make_case(case_id, family, source, context, evidence, answer, keywords):
    return {
        "case_id": case_id,
        "family": family,
        "source": source,
        "context": context,
        "evidence": [
            {"id": f"E{i+1}", "text": text} for i, text in enumerate(evidence)
        ],
        "answer": answer,
        "answer_keywords": keywords,
    }

def full_answer(facts, assumptions, hypotheses, next_test, conclusion,
                confidence, unknowns):
    return {
        "facts": facts,
        "assumptions": assumptions,
        "hypotheses": hypotheses,
        "best_next_test": next_test,
        "conclusion": {
            "claim": conclusion,
            "support": sorted({
                ref for hypothesis in hypotheses[:1]
                for ref in hypothesis.get("support", [])
            }),
            "confidence": confidence,
        },
        "unknowns": unknowns,
    }

## 3. Curated Holmes case reconstructions

These are concise, original paraphrases of reasoning structures in six
public-domain stories. They are not verbatim story passages. A production
dataset would require independent expert annotation and adjudication.

In [ ]:
HOLMES_CASES = [
    make_case(
        "holmes_silver_blaze", "familiar_access",
        "Silver Blaze — curated paraphrase",
        "A racehorse disappears from a guarded stable during the night.",
        [
            "The stable dog did not bark during the intrusion.",
            "The stable boy had been drugged.",
            "The horse was removed without a broken door.",
            "A visitor unfamiliar to the dog had recently been near the stable.",
        ],
        full_answer(
            [
                {"evidence_id": "E1", "statement": "The dog remained silent."},
                {"evidence_id": "E2", "statement": "The stable boy was drugged."},
                {"evidence_id": "E3", "statement": "The door was not forced."},
            ],
            ["Silence proves that nobody entered."],
            [
                {
                    "id": "H1",
                    "claim": "A person familiar to the dog removed the horse.",
                    "support": ["E1", "E3"],
                    "contradictions": [],
                    "prediction": "The responsible person had routine access to the stable.",
                },
                {
                    "id": "H2",
                    "claim": "An unfamiliar outsider forced entry.",
                    "support": ["E4"],
                    "contradictions": ["E1", "E3"],
                    "prediction": "There should be forced-entry traces or an alarm from the dog.",
                },
            ],
            {
                "test": "List people known to the dog and check their movements.",
                "distinguishes": ["H1", "H2"],
                "rationale": "Familiar access explains both silence and the intact door.",
            },
            "The intruder was probably familiar to the dog and stable.",
            0.88,
            ["The precise motive and present location of the horse."],
        ),
        ["familiar", "dog", "access"],
    ),
    make_case(
        "holmes_red_headed_league", "staged_distraction",
        "The Red-Headed League — curated paraphrase",
        "A pawnbroker is paid unusually well to copy an encyclopedia away from his shop.",
        [
            "The job requires the pawnbroker to leave his shop at fixed hours.",
            "The employer closely inspected whether his hair met an arbitrary condition.",
            "The assistant accepts unusually low wages.",
            "The shop is near a bank.",
            "The supposed organization suddenly dissolves.",
        ],
        full_answer(
            [{"evidence_id": f"E{i}", "statement": text} for i, text in enumerate([
                "The job creates predictable absence.", "The condition is contrived.",
                "The assistant has another incentive.", "A bank is nearby.",
                "The organization disappears abruptly."
            ], 1)],
            ["The copying task has intrinsic commercial value."],
            [
                {
                    "id": "H1",
                    "claim": "The job is a diversion that removes the owner from the shop.",
                    "support": ["E1", "E2", "E3", "E5"],
                    "contradictions": [],
                    "prediction": "Activity connected to the assistant occurs during the absence.",
                },
                {
                    "id": "H2",
                    "claim": "The organization genuinely values copying work.",
                    "support": [],
                    "contradictions": ["E2", "E5"],
                    "prediction": "The organization should have stable premises and a durable purpose.",
                },
            ],
            {
                "test": "Inspect the cellar and the route between the shop and bank.",
                "distinguishes": ["H1", "H2"],
                "rationale": "A diversion predicts hidden activity at the unattended premises.",
            },
            "The employment scheme is probably a staged diversion for a crime near the shop.",
            0.93,
            ["The exact method and timing of the planned crime."],
        ),
        ["diversion", "shop", "bank"],
    ),
    make_case(
        "holmes_speckled_band", "mechanical_route",
        "The Speckled Band — curated paraphrase",
        "A woman dies in a locked bedroom after reporting a whistle at night.",
        [
            "The bed is fixed to the floor.",
            "A bell rope hangs beside the bed but is not connected to a bell.",
            "A ventilator opens into the adjacent room rather than outside.",
            "A low whistle is heard at night.",
            "The adjacent occupant has access to unusual animals.",
        ],
        full_answer(
            [
                {"evidence_id": "E1", "statement": "The bed cannot be moved."},
                {"evidence_id": "E2", "statement": "The bell rope is nonfunctional."},
                {"evidence_id": "E3", "statement": "The ventilator connects the rooms."},
                {"evidence_id": "E4", "statement": "A whistle occurs at night."},
            ],
            ["A locked door means no external mechanism can act."],
            [
                {
                    "id": "H1",
                    "claim": "A controlled hazard travels from the adjacent room to the bed.",
                    "support": ["E1", "E2", "E3", "E4", "E5"],
                    "contradictions": [],
                    "prediction": "The rope and ventilator form a usable route.",
                },
                {
                    "id": "H2",
                    "claim": "A stranger enters through the bedroom door.",
                    "support": [],
                    "contradictions": ["E3"],
                    "prediction": "The lock or doorway should show access traces.",
                },
            ],
            {
                "test": "Observe the rope and ventilator safely during the expected whistle.",
                "distinguishes": ["H1", "H2"],
                "rationale": "The route hypothesis predicts a repeatable event at that location.",
            },
            "The room arrangement likely delivers a controlled hazard from next door.",
            0.90,
            ["The exact nature of the hazard until directly observed."],
        ),
        ["adjacent", "route", "ventilator"],
    ),
    make_case(
        "holmes_blue_carbuncle", "object_chain",
        "The Blue Carbuncle — curated paraphrase",
        "A valuable jewel is found inside a holiday goose acquired through ordinary trade.",
        [
            "The current owner bought the goose through a dealer.",
            "The dealer obtained it from a supplier.",
            "The jewel could not have entered the goose after it was cooked.",
            "A person connected to the jewel had access before the goose was sold.",
        ],
        full_answer(
            [
                {"evidence_id": "E1", "statement": "The goose passed through a dealer."},
                {"evidence_id": "E2", "statement": "The dealer had a supplier."},
                {"evidence_id": "E3", "statement": "The jewel entered before cooking."},
            ],
            ["The current owner must have hidden the jewel."],
            [
                {
                    "id": "H1",
                    "claim": "The jewel entered the goose earlier in the supply chain.",
                    "support": ["E1", "E2", "E3", "E4"],
                    "contradictions": [],
                    "prediction": "Tracing the supplier reveals a link to the jewel.",
                },
                {
                    "id": "H2",
                    "claim": "The current owner inserted the jewel after purchase.",
                    "support": [],
                    "contradictions": ["E3"],
                    "prediction": "There should be a feasible post-purchase insertion method.",
                },
            ],
            {
                "test": "Trace the goose backward from dealer to supplier.",
                "distinguishes": ["H1", "H2"],
                "rationale": "A chain-of-custody reconstruction localizes when insertion occurred.",
            },
            "The jewel was probably concealed before the goose reached its current owner.",
            0.86,
            ["Which upstream person performed the concealment."],
        ),
        ["supply", "chain", "before"],
    ),
    make_case(
        "holmes_dancing_men", "coded_pattern",
        "The Dancing Men — curated paraphrase",
        "Repeated sequences of unusual stick figures appear around a country house.",
        [
            "Several symbol groups repeat in different messages.",
            "One short group appears where a name would plausibly occur.",
            "A recurring terminal symbol behaves like punctuation.",
            "The recipient reacts with fear rather than confusion.",
        ],
        full_answer(
            [
                {"evidence_id": "E1", "statement": "Symbol groups repeat."},
                {"evidence_id": "E2", "statement": "A short group may be a name."},
                {"evidence_id": "E3", "statement": "A terminal symbol may mark boundaries."},
                {"evidence_id": "E4", "statement": "The recipient recognizes the threat."},
            ],
            ["Every distinct figure necessarily represents a complete word."],
            [
                {
                    "id": "H1",
                    "claim": "The figures encode language with substitution and boundaries.",
                    "support": ["E1", "E2", "E3", "E4"],
                    "contradictions": [],
                    "prediction": "Frequency and repeated-pattern analysis yields consistent letters.",
                },
                {
                    "id": "H2",
                    "claim": "The figures are random decoration.",
                    "support": [],
                    "contradictions": ["E1", "E4"],
                    "prediction": "No stable mapping should decode multiple messages.",
                },
            ],
            {
                "test": "Test a substitution mapping against every repeated group.",
                "distinguishes": ["H1", "H2"],
                "rationale": "A real code should decode consistently across messages.",
            },
            "The figures are probably a substitution code directed at the recipient.",
            0.91,
            ["The full key until more messages are analyzed."],
        ),
        ["code", "substitution", "pattern"],
    ),
    make_case(
        "holmes_six_napoleons", "selective_search",
        "The Six Napoleons — curated paraphrase",
        "Several identical plaster busts are stolen and smashed in separate places.",
        [
            "Only busts from the same production batch are targeted.",
            "The fragments are examined rather than merely vandalized.",
            "Other valuable objects are ignored.",
            "The incidents follow the distribution path of the busts.",
        ],
        full_answer(
            [
                {"evidence_id": "E1", "statement": "One production batch is targeted."},
                {"evidence_id": "E2", "statement": "Fragments are searched."},
                {"evidence_id": "E3", "statement": "Other valuables are ignored."},
                {"evidence_id": "E4", "statement": "Events follow distribution."},
            ],
            ["The attacker has a general hatred of the historical figure."],
            [
                {
                    "id": "H1",
                    "claim": "Someone is searching the busts for a concealed object.",
                    "support": ["E1", "E2", "E3", "E4"],
                    "contradictions": [],
                    "prediction": "The remaining busts from the batch will be targeted.",
                },
                {
                    "id": "H2",
                    "claim": "The acts are indiscriminate ideological vandalism.",
                    "support": [],
                    "contradictions": ["E1", "E2", "E3"],
                    "prediction": "Targets should include unrelated busts and locations.",
                },
            ],
            {
                "test": "Locate and protect the remaining busts from the batch.",
                "distinguishes": ["H1", "H2"],
                "rationale": "Selective search predicts the next targets.",
            },
            "The busts are being broken to recover something hidden in one of them.",
            0.94,
            ["Which bust contains the object and who concealed it."],
        ),
        ["hidden", "search", "bust"],
    ),
]

print(f"Curated cases: {len(HOLMES_CASES)}")
print(json.dumps(HOLMES_CASES[0], indent=2)[:4500])

## 4. Procedural curriculum

Fine-tuning on six stories would memorize them. The generator below creates
varied cases that preserve abstract investigation patterns while changing
people, places, objects, timings, and distractors.

The held-out families, `dual_route` and `access_log`, are never included in
training.

In [ ]:
PEOPLE = ["Avery", "Blake", "Casey", "Devon", "Emery", "Finley", "Gray", "Harper"]
PLACES = ["archive", "laboratory", "gallery", "warehouse", "trading room", "library"]
OBJECTS = ["sealed ledger", "prototype", "rare map", "backup drive", "signed contract"]
DISTRACTORS = [
    "A window was open, but weather records show strong wind.",
    "A visitor wore a bright coat unrelated to the event.",
    "A broken cup was found in a different room.",
    "A newspaper carried an alarming headline that morning.",
]

TRAIN_FAMILIES = [
    "familiar_access",
    "staged_distraction",
    "mechanical_route",
    "object_chain",
    "coded_pattern",
    "selective_search",
]
HOLDOUT_FAMILIES = ["dual_route", "access_log"]

def fact(evidence_id, statement):
    return {"evidence_id": evidence_id, "statement": statement}

def hypothesis(hid, claim, support, contradictions, prediction):
    return {
        "id": hid,
        "claim": claim,
        "support": support,
        "contradictions": contradictions,
        "prediction": prediction,
    }

def generated_case(family, index, rng):
    person = rng.choice(PEOPLE)
    second = rng.choice([p for p in PEOPLE if p != person])
    place = rng.choice(PLACES)
    obj = rng.choice(OBJECTS)
    distractor = rng.choice(DISTRACTORS)
    cid = f"synthetic_{family}_{index:04d}"

    if family == "familiar_access":
        evidence = [
            "The alarm did not activate during the incident.",
            "The outer lock shows no damage.",
            f"{person} had routine authorized access to the {place}.",
            f"The {obj} disappeared during a narrow time window.",
            distractor,
        ]
        hypotheses = [
            hypothesis("H1", "An authorized insider used familiar access.",
                       ["E1", "E2", "E3", "E4"], [], "Authorized-access records should narrow the suspects."),
            hypothesis("H2", "An unknown outsider forced entry.",
                       ["E4"], ["E1", "E2"], "Forced-entry or alarm traces should exist."),
        ]
        conclusion = "An authorized insider is more likely than a forced-entry outsider."
        keywords = ["authorized", "insider", "access"]
        next_test = {
            "test": "Compare authorized-access records with the disappearance window.",
            "distinguishes": ["H1", "H2"],
            "rationale": "The hypotheses predict different access traces.",
        }
    elif family == "staged_distraction":
        evidence = [
            f"A noisy incident drew staff away from the {place}.",
            f"The {obj} vanished during that exact interval.",
            f"{person} knew the emergency procedure.",
            "The noisy incident caused little actual damage.",
            distractor,
        ]
        hypotheses = [
            hypothesis("H1", "The noisy incident was a deliberate diversion.",
                       ["E1", "E2", "E3", "E4"], [], "Its initiator should connect to the target or timing."),
            hypothesis("H2", "The incident and disappearance were independent accidents.",
                       ["E1", "E2"], ["E4"], "No planning link should be found."),
        ]
        conclusion = "The low-damage incident was probably staged to create access."
        keywords = ["diversion", "staged", "access"]
        next_test = {
            "test": "Identify who initiated the incident and compare that with target knowledge.",
            "distinguishes": ["H1", "H2"],
            "rationale": "A diversion requires coordination with the theft window.",
        }
    elif family == "mechanical_route":
        evidence = [
            f"The {place} door remained locked.",
            "A narrow service conduit links the room to an adjacent area.",
            f"The {obj} was positioned directly beneath the conduit.",
            f"{person} controlled the adjacent area.",
            distractor,
        ]
        hypotheses = [
            hypothesis("H1", "The conduit was used as an indirect route.",
                       ["E1", "E2", "E3", "E4"], [], "The conduit should carry matching trace evidence."),
            hypothesis("H2", "Someone entered through the locked door.",
                       ["E1"], ["E1", "E2"], "The lock should show bypass evidence."),
        ]
        conclusion = "An indirect route through the adjacent area best explains the locked room."
        keywords = ["indirect", "route", "adjacent"]
        next_test = {
            "test": "Inspect the conduit for fibers, residue, or tool marks.",
            "distinguishes": ["H1", "H2"],
            "rationale": "Only the indirect-route hypothesis predicts conduit traces.",
        }
    elif family == "object_chain":
        evidence = [
            f"The {obj} passed from {person} to a broker and then to {second}.",
            "The anomaly was already sealed inside the packaging at final delivery.",
            "The broker logged no package opening.",
            f"{person} controlled the item before the seal was applied.",
            distractor,
        ]
        hypotheses = [
            hypothesis("H1", "The anomaly entered before the broker received the item.",
                       ["E1", "E2", "E3", "E4"], [], "Upstream records should reveal the insertion opportunity."),
            hypothesis("H2", "The final recipient altered the sealed package.",
                       ["E2"], ["E2", "E3"], "The seal should show post-delivery tampering."),
        ]
        conclusion = "The anomaly most likely entered upstream before the package was sealed."
        keywords = ["upstream", "before", "sealed"]
        next_test = {
            "test": "Reconstruct custody before sealing and inspect the seal for tampering.",
            "distinguishes": ["H1", "H2"],
            "rationale": "The alternatives imply different insertion times.",
        }
    elif family == "coded_pattern":
        evidence = [
            "Several symbol groups recur across separate messages.",
            "One separator appears at consistent boundaries.",
            "Substituting the most frequent symbol yields plausible fragments.",
            f"{person} recognizes one repeated group.",
            distractor,
        ]
        hypotheses = [
            hypothesis("H1", "The messages use a consistent substitution code.",
                       ["E1", "E2", "E3", "E4"], [], "One mapping should decode every occurrence consistently."),
            hypothesis("H2", "The symbols are random marks.",
                       [], ["E1", "E2", "E3"], "No stable mapping should generalize."),
        ]
        conclusion = "The repeated structure supports a consistent substitution code."
        keywords = ["substitution", "code", "consistent"]
        next_test = {
            "test": "Apply one candidate mapping to all messages and count inconsistencies.",
            "distinguishes": ["H1", "H2"],
            "rationale": "A real code should generalize across messages.",
        }
    elif family == "selective_search":
        evidence = [
            f"Only copies of the same {obj} edition were damaged.",
            "Fragments were arranged as if examined.",
            "Nearby valuables were untouched.",
            f"{person} obtained the distribution list.",
            distractor,
        ]
        hypotheses = [
            hypothesis("H1", "Someone is searching this batch for a concealed item.",
                       ["E1", "E2", "E3", "E4"], [], "Undamaged copies from the batch will be targeted."),
            hypothesis("H2", "The damage is indiscriminate vandalism.",
                       [], ["E1", "E2", "E3"], "Targets should not follow one batch."),
        ]
        conclusion = "The selective damage is a search for something concealed in one batch."
        keywords = ["search", "concealed", "batch"]
        next_test = {
            "test": "Locate and monitor the remaining copies from the same batch.",
            "distinguishes": ["H1", "H2"],
            "rationale": "Selective search predicts the next targets.",
        }
    elif family == "dual_route":
        evidence = [
            f"Wet clay appears on only one side of the {place} floor.",
            "The main corridor was dry throughout the period.",
            "A maintenance stair opens onto the wet side.",
            f"{person} claimed to use only the main corridor.",
            distractor,
        ]
        hypotheses = [
            hypothesis("H1", "The maintenance stair was used.",
                       ["E1", "E2", "E3", "E4"], [], "Clay on the stair should match the floor trace."),
            hypothesis("H2", "The main corridor was used.",
                       ["E4"], ["E1", "E2"], "The main corridor should contain matching clay."),
        ]
        conclusion = "The physical trace indicates use of the maintenance stair."
        keywords = ["maintenance", "stair", "trace"]
        next_test = {
            "test": "Compare clay samples from the floor and maintenance stair.",
            "distinguishes": ["H1", "H2"],
            "rationale": "The route hypotheses predict different trace locations.",
        }
    elif family == "access_log":
        evidence = [
            f"The digital log assigns an entry to {person}.",
            f"Camera footage shows {person} elsewhere at that time.",
            "A shared service credential can create the same log label.",
            f"{second} used that service credential earlier that day.",
            distractor,
        ]
        hypotheses = [
            hypothesis("H1", "A shared credential created a misleading attribution.",
                       ["E2", "E3", "E4"], [], "Low-level authentication records should identify the device."),
            hypothesis("H2", f"{person} physically entered as the summary log states.",
                       ["E1"], ["E2"], "Camera or device evidence should place the person there."),
        ]
        conclusion = "The summary log is likely a credential attribution, not proof of physical presence."
        keywords = ["credential", "attribution", "log"]
        next_test = {
            "test": "Inspect device-level authentication records behind the summary log.",
            "distinguishes": ["H1", "H2"],
            "rationale": "The same display label can arise from different devices.",
        }
    else:
        raise ValueError(f"Unknown family: {family}")

    facts = [
        fact(f"E{i+1}", statement) for i, statement in enumerate(evidence[:4])
    ]
    answer = full_answer(
        facts=facts,
        assumptions=["The most vivid clue is necessarily the most relevant."],
        hypotheses=hypotheses,
        next_test=next_test,
        conclusion=conclusion,
        confidence=round(rng.uniform(0.74, 0.91), 2),
        unknowns=["Motive and exact sequence remain partly uncertain."],
    )
    return make_case(
        cid, family, "procedural curriculum", 
        f"Investigate an anomaly involving a {obj} in a {place}.",
        evidence, answer, keywords
    )

def build_case_pool(cases_per_family, seed=SEED):
    rng = random.Random(seed)
    pool = []
    for family in TRAIN_FAMILIES + HOLDOUT_FAMILIES:
        for i in range(cases_per_family):
            pool.append(generated_case(family, i, rng))
    return pool

synthetic_cases = build_case_pool(CFG["cases_per_family"])
print(f"Synthetic cases: {len(synthetic_cases)}")
print("Families:", sorted({c["family"] for c in synthetic_cases}))

## 5. Split by case before expansion

All tasks derived from a case remain in one split. This prevents an easy form
of leakage where the model sees the same facts under a different task label.
The OOD set contains families never seen in training.

In [ ]:
def split_cases(cases, val_fraction=0.10, seed=SEED):
    rng = random.Random(seed)
    train_candidates = [c for c in cases if c["family"] in TRAIN_FAMILIES]
    ood_cases = [c for c in cases if c["family"] in HOLDOUT_FAMILIES]

    by_family = defaultdict(list)
    for case in train_candidates:
        by_family[case["family"]].append(case)

    train_cases, val_cases = [], []
    for family, family_cases in by_family.items():
        rng.shuffle(family_cases)
        n_val = max(1, round(len(family_cases) * val_fraction))
        val_cases.extend(family_cases[:n_val])
        train_cases.extend(family_cases[n_val:])

    # Four curated cases support training; two remain qualitative transfer tests.
    train_cases.extend(HOLMES_CASES[:4])
    ood_cases.extend(HOLMES_CASES[4:])
    rng.shuffle(train_cases)
    rng.shuffle(val_cases)
    rng.shuffle(ood_cases)
    return train_cases, val_cases, ood_cases

train_cases, val_cases, ood_cases = split_cases(synthetic_cases)
strict_ood_cases = [
    c for c in ood_cases if c["family"] in HOLDOUT_FAMILIES
]
qualitative_transfer_cases = [
    c for c in ood_cases if c["family"] not in HOLDOUT_FAMILIES
]

train_ids = {c["case_id"] for c in train_cases}
val_ids = {c["case_id"] for c in val_cases}
ood_ids = {c["case_id"] for c in ood_cases}
assert train_ids.isdisjoint(val_ids)
assert train_ids.isdisjoint(ood_ids)
assert val_ids.isdisjoint(ood_ids)
assert not ({c["family"] for c in train_cases} & set(HOLDOUT_FAMILIES))

print("Train cases:", len(train_cases))
print("Validation cases:", len(val_cases))
print("Strict OOD cases:", len(strict_ood_cases))
print("Strict OOD families:", sorted({c["family"] for c in strict_ood_cases}))
print("Qualitative held-back Holmes cases:", len(qualitative_transfer_cases))

In [ ]:
def case_to_user_prompt(case, task):
    evidence_block = "\n".join(
        f'{item["id"]}: {item["text"]}' for item in case["evidence"]
    )
    return (
        f"TASK: {task}\n"
        f"CASE CONTEXT: {case['context']}\n"
        f"EVIDENCE:\n{evidence_block}\n"
        f"OUTPUT CONTRACT: {TASK_INSTRUCTIONS[task]}"
    )

def target_for_task(case, task):
    answer = case["answer"]
    if task == "OBSERVATION_AUDIT":
        return {
            "facts": answer["facts"],
            "assumptions": answer["assumptions"],
        }
    if task == "HYPOTHESIS_TEST":
        return {
            "hypotheses": answer["hypotheses"],
            "best_next_test": answer["best_next_test"],
        }
    if task == "FULL_ANALYSIS":
        return answer
    raise ValueError(task)

def expand_curriculum(cases):
    rows = []
    for case in cases:
        for task in ["OBSERVATION_AUDIT", "HYPOTHESIS_TEST", "FULL_ANALYSIS"]:
            rows.append({
                "case_id": case["case_id"],
                "family": case["family"],
                "task": task,
                "messages": [
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": case_to_user_prompt(case, task)},
                    {
                        "role": "assistant",
                        "content": json.dumps(
                            target_for_task(case, task),
                            ensure_ascii=False,
                            separators=(",", ":"),
                        ),
                    },
                ],
            })
    return rows

train_rows = expand_curriculum(train_cases)
val_rows = expand_curriculum(val_cases)
print("Training examples:", len(train_rows))
print("Validation examples:", len(val_rows))
print(json.dumps(train_rows[0], indent=2)[:5000])

## 6. Load the base model in 4 bits and attach LoRA

The pretrained model supplies language competence. LoRA trains a small set of
adapter parameters. Quantization keeps the frozen base weights compact enough
for a T4 GPU.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map={"": 0},
    torch_dtype=COMPUTE_DTYPE,
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

try:
    model.gradient_checkpointing_enable(
        gradient_checkpointing_kwargs={"use_reentrant": False}
    )
except TypeError:
    model.gradient_checkpointing_enable()

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 7. Prompt-masked supervised fine-tuning

Loss is calculated only on the assistant answer—not on the system instruction
or case prompt. Long prompts are truncated from the left only when necessary,
preserving the complete target whenever possible.

In [ ]:
def encode_row(row, tokenizer, max_length):
    prompt_messages = row["messages"][:2]
    target_text = row["messages"][2]["content"] + tokenizer.eos_token
    prompt_text = tokenizer.apply_chat_template(
        prompt_messages, tokenize=False, add_generation_prompt=True
    )
    prompt_ids = tokenizer(
        prompt_text, add_special_tokens=False
    )["input_ids"]
    target_ids = tokenizer(
        target_text, add_special_tokens=False
    )["input_ids"]

    # Preserve the supervised answer; remove oldest prompt tokens if needed.
    if len(target_ids) >= max_length:
        target_ids = target_ids[:max_length]
        prompt_ids = []
    else:
        prompt_budget = max_length - len(target_ids)
        prompt_ids = prompt_ids[-prompt_budget:]

    input_ids = prompt_ids + target_ids
    labels = [-100] * len(prompt_ids) + target_ids
    return {
        "input_ids": input_ids,
        "attention_mask": [1] * len(input_ids),
        "labels": labels,
        "case_id": row["case_id"],
        "task": row["task"],
    }

class ReasoningDataset(Dataset):
    def __init__(self, rows, tokenizer, max_length):
        self.items = [encode_row(r, tokenizer, max_length) for r in rows]

    def __len__(self):
        return len(self.items)

    def __getitem__(self, index):
        return self.items[index]

@dataclass
class CausalCollator:
    pad_token_id: int

    def __call__(self, features):
        max_len = max(len(f["input_ids"]) for f in features)
        batch = {"input_ids": [], "attention_mask": [], "labels": []}
        for feature in features:
            pad_n = max_len - len(feature["input_ids"])
            batch["input_ids"].append(
                feature["input_ids"] + [self.pad_token_id] * pad_n
            )
            batch["attention_mask"].append(
                feature["attention_mask"] + [0] * pad_n
            )
            batch["labels"].append(
                feature["labels"] + [-100] * pad_n
            )
        return {k: torch.tensor(v, dtype=torch.long) for k, v in batch.items()}

train_dataset = ReasoningDataset(train_rows, tokenizer, MAX_LENGTH)
val_dataset = ReasoningDataset(val_rows, tokenizer, MAX_LENGTH)
collator = CausalCollator(tokenizer.pad_token_id)

train_loader = DataLoader(
    train_dataset,
    batch_size=MICRO_BATCH_SIZE,
    shuffle=True,
    collate_fn=collator,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=MICRO_BATCH_SIZE,
    shuffle=False,
    collate_fn=collator,
)

lengths = [len(item["input_ids"]) for item in train_dataset.items]
print("Token lengths: min/median/max =",
      min(lengths), int(np.median(lengths)), max(lengths))
print("Truncated examples:", sum(length == MAX_LENGTH for length in lengths))
assert all(any(label != -100 for label in item["labels"])
           for item in train_dataset.items)

## 8. Generation helpers and an unadapted baseline

The baseline is captured with the LoRA adapter disabled. Use deterministic
decoding for evaluation so comparisons are reproducible.

In [ ]:
def prompt_inputs(case, task="FULL_ANALYSIS"):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": case_to_user_prompt(case, task)},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    return tokenizer(text, return_tensors="pt").to("cuda")

@torch.inference_mode()
def generate_case(model, case, max_new_tokens=420, adapter_enabled=True):
    model.eval()
    inputs = prompt_inputs(case)
    context = (
        model.disable_adapter()
        if hasattr(model, "disable_adapter") and not adapter_enabled
        else None
    )
    if context is None:
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    else:
        with context:
            output = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )
    generated = output[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

baseline_cases = strict_ood_cases[:min(3, len(strict_ood_cases))]
baseline_outputs = {}
for case in baseline_cases:
    text = generate_case(model, case, adapter_enabled=False)
    baseline_outputs[case["case_id"]] = text
    print("\nCASE:", case["case_id"])
    print(text[:1800])

## 9. QLoRA training loop

This explicit loop keeps the mechanics visible: gradient accumulation,
clipping, cosine learning-rate decay, periodic validation, and a hard optimizer
step ceiling.

In [ ]:
trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(
    trainable_params,
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM_STEPS)
planned_steps = min(
    CFG["max_optimizer_steps"],
    steps_per_epoch * CFG["epochs"],
)
warmup_steps = max(1, round(0.05 * planned_steps))
scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=planned_steps,
)

@torch.inference_mode()
def evaluate_loss(model, loader, max_batches=32):
    model.eval()
    losses = []
    for batch_index, batch in enumerate(loader):
        if batch_index >= max_batches:
            break
        batch = {k: v.to("cuda") for k, v in batch.items()}
        loss = model(**batch).loss
        losses.append(float(loss.detach().cpu()))
    model.train()
    return float(np.mean(losses)) if losses else float("nan")

history = {"optimizer_step": [], "train_loss": [], "val_loss": [], "lr": []}
optimizer.zero_grad(set_to_none=True)
optimizer_step = 0
running_loss = 0.0
micro_steps = 0
start_time = time.time()
model.train()

for epoch in range(CFG["epochs"]):
    for batch_index, batch in enumerate(train_loader):
        batch = {k: v.to("cuda") for k, v in batch.items()}
        outputs = model(**batch)
        raw_loss = outputs.loss
        (raw_loss / GRAD_ACCUM_STEPS).backward()
        running_loss += float(raw_loss.detach().cpu())
        micro_steps += 1

        should_step = (
            micro_steps % GRAD_ACCUM_STEPS == 0
            or batch_index == len(train_loader) - 1
        )
        if not should_step:
            continue

        torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad(set_to_none=True)
        optimizer_step += 1

        mean_train_loss = running_loss / micro_steps
        if optimizer_step == 1 or optimizer_step % 10 == 0:
            val_loss = evaluate_loss(
                model, val_loader,
                max_batches=8 if RUN_MODE == "smoke" else 24
            )
            history["optimizer_step"].append(optimizer_step)
            history["train_loss"].append(mean_train_loss)
            history["val_loss"].append(val_loss)
            history["lr"].append(scheduler.get_last_lr()[0])
            elapsed = (time.time() - start_time) / 60
            print(
                f"epoch={epoch+1} step={optimizer_step}/{planned_steps} "
                f"train={mean_train_loss:.4f} val={val_loss:.4f} "
                f"minutes={elapsed:.1f}"
            )

        running_loss = 0.0
        micro_steps = 0
        if optimizer_step >= planned_steps:
            break
    if optimizer_step >= planned_steps:
        break

print(f"Training completed in {(time.time()-start_time)/60:.1f} minutes.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history["optimizer_step"], history["train_loss"], label="train")
axes[0].plot(history["optimizer_step"], history["val_loss"], label="validation")
axes[0].set_title("Masked causal-LM loss")
axes[0].set_xlabel("Optimizer step")
axes[0].set_ylabel("Loss")
axes[0].legend()

axes[1].plot(history["optimizer_step"], history["lr"])
axes[1].set_title("Learning-rate schedule")
axes[1].set_xlabel("Optimizer step")
axes[1].set_ylabel("Learning rate")
plt.tight_layout()
plt.show()

## 10. Deterministic verifier

The verifier does not decide whether a conclusion is ultimately true. It checks
whether the response is parseable, follows the contract, references only
available evidence, considers alternatives, and expresses bounded confidence.

In [ ]:
def parse_json_output(text):
    cleaned = text.strip()
    cleaned = re.sub(r"^```(?:json)?\s*", "", cleaned)
    cleaned = re.sub(r"\s*```$", "", cleaned)
    try:
        return json.loads(cleaned), None
    except json.JSONDecodeError:
        start, end = cleaned.find("{"), cleaned.rfind("}")
        if start >= 0 and end > start:
            try:
                return json.loads(cleaned[start:end+1]), None
            except json.JSONDecodeError as exc:
                return None, str(exc)
        return None, "No JSON object found."

def verify_output(case, text):
    obj, parse_error = parse_json_output(text)
    report = {
        "json_valid": obj is not None,
        "schema_complete": False,
        "evidence_reference_precision": 0.0,
        "has_competing_hypotheses": False,
        "confidence_valid": False,
        "errors": [],
    }
    if obj is None:
        report["errors"].append(parse_error)
        report["score"] = 0.0
        return report

    keys = set(obj)
    report["schema_complete"] = REQUIRED_FULL_KEYS.issubset(keys)
    if not report["schema_complete"]:
        report["errors"].append(
            f"Missing keys: {sorted(REQUIRED_FULL_KEYS - keys)}"
        )

    allowed = {item["id"] for item in case["evidence"]}
    refs = []
    for item in obj.get("facts", []):
        if isinstance(item, dict) and "evidence_id" in item:
            refs.append(item["evidence_id"])
    hypotheses = obj.get("hypotheses", [])
    if isinstance(hypotheses, list):
        report["has_competing_hypotheses"] = len(hypotheses) >= 2
        for hyp in hypotheses:
            if isinstance(hyp, dict):
                refs.extend(hyp.get("support", []))
                refs.extend(hyp.get("contradictions", []))
    conclusion = obj.get("conclusion", {})
    if isinstance(conclusion, dict):
        refs.extend(conclusion.get("support", []))
        confidence = conclusion.get("confidence")
        report["confidence_valid"] = (
            isinstance(confidence, (int, float))
            and 0.0 <= float(confidence) <= 1.0
        )
    valid_refs = [ref for ref in refs if ref in allowed]
    report["evidence_reference_precision"] = (
        len(valid_refs) / len(refs) if refs else 0.0
    )
    invalid = sorted(set(refs) - allowed)
    if invalid:
        report["errors"].append(f"Invented evidence IDs: {invalid}")

    components = [
        report["json_valid"],
        report["schema_complete"],
        report["has_competing_hypotheses"],
        report["confidence_valid"],
        report["evidence_reference_precision"],
    ]
    report["score"] = float(np.mean(components))
    return report

def keyword_score(case, text):
    low = text.lower()
    words = case["answer_keywords"]
    return sum(word.lower() in low for word in words) / max(1, len(words))

## 11. Before-and-after benchmark

The main comparison measures formatting and evidence discipline. Keyword score
is only a rough proxy for conclusion content; it is not a substitute for expert
judgment.

In [ ]:
eval_cases = strict_ood_cases[:min(CFG["eval_cases"], len(strict_ood_cases))]
benchmark_rows = []

for i, case in enumerate(eval_cases, 1):
    print(f"Generating {i}/{len(eval_cases)}: {case['case_id']}")
    base_text = baseline_outputs.get(case["case_id"])
    if base_text is None:
        base_text = generate_case(model, case, adapter_enabled=False)
    tuned_text = generate_case(model, case, adapter_enabled=True)

    for label, text in [("base", base_text), ("holmesian_lora", tuned_text)]:
        verification = verify_output(case, text)
        benchmark_rows.append({
            "case_id": case["case_id"],
            "family": case["family"],
            "model": label,
            "verifier_score": verification["score"],
            "json_valid": float(verification["json_valid"]),
            "schema_complete": float(verification["schema_complete"]),
            "evidence_precision": verification["evidence_reference_precision"],
            "keyword_score": keyword_score(case, text),
            "output": text,
        })

def summarize(rows, model_label):
    selected = [r for r in rows if r["model"] == model_label]
    metrics = [
        "verifier_score", "json_valid", "schema_complete",
        "evidence_precision", "keyword_score"
    ]
    return {
        metric: round(float(np.mean([r[metric] for r in selected])), 3)
        for metric in metrics
    }

summary = {
    "base": summarize(benchmark_rows, "base"),
    "holmesian_lora": summarize(benchmark_rows, "holmesian_lora"),
}
print(json.dumps(summary, indent=2))

In [ ]:
metrics = ["verifier_score", "json_valid", "schema_complete",
           "evidence_precision", "keyword_score"]
x = np.arange(len(metrics))
width = 0.36

fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(x - width/2, [summary["base"][m] for m in metrics],
       width, label="Base")
ax.bar(x + width/2, [summary["holmesian_lora"][m] for m in metrics],
       width, label="Holmesian LoRA")
ax.set_xticks(x)
ax.set_xticklabels(metrics, rotation=20, ha="right")
ax.set_ylim(0, 1.05)
ax.set_ylabel("Mean score")
ax.set_title("Held-out family benchmark")
ax.legend()
plt.tight_layout()
plt.show()

## 12. Counterfactual robustness probe

A useful reasoner should revise confidence when decisive evidence disappears.
This probe removes the strongest clue from a held-out case and compares the
reported conclusion confidence. The desired direction is lower confidence, but
a small model may fail this test.

In [ ]:
def extract_confidence(text):
    obj, _ = parse_json_output(text)
    if not isinstance(obj, dict):
        return None
    conclusion = obj.get("conclusion", {})
    value = conclusion.get("confidence") if isinstance(conclusion, dict) else None
    return float(value) if isinstance(value, (int, float)) else None

probe_case = deepcopy(eval_cases[0])
full_output = generate_case(model, probe_case, adapter_enabled=True)

ablated_case = deepcopy(probe_case)
removed = ablated_case["evidence"].pop(0)
ablated_output = generate_case(model, ablated_case, adapter_enabled=True)

print("Removed evidence:", removed)
print("Full-evidence confidence:", extract_confidence(full_output))
print("Ablated-evidence confidence:", extract_confidence(ablated_output))
print("\nFull output:\n", full_output[:2500])
print("\nAblated output:\n", ablated_output[:2500])

## 13. Prepare preference pairs for a later DPO stage

The deterministic verifier can create weak preference data: a grounded gold
answer is preferred over a deliberately corrupted answer containing invented
evidence, premature certainty, or only one hypothesis. This notebook exports
the pairs but does not run preference optimization by default.

In [ ]:
def corrupt_answer(case, rng):
    bad = deepcopy(case["answer"])
    corruption = rng.choice(["invented_evidence", "one_hypothesis", "overconfidence"])
    if corruption == "invented_evidence":
        bad["conclusion"]["support"] = ["E999"]
    elif corruption == "one_hypothesis":
        bad["hypotheses"] = bad["hypotheses"][:1]
    else:
        bad["conclusion"]["confidence"] = 1.0
        bad["unknowns"] = []
    return bad, corruption

rng = random.Random(SEED)
preference_pairs = []
for case in train_cases:
    rejected, corruption = corrupt_answer(case, rng)
    preference_pairs.append({
        "case_id": case["case_id"],
        "prompt": case_to_user_prompt(case, "FULL_ANALYSIS"),
        "chosen": json.dumps(case["answer"], ensure_ascii=False),
        "rejected": json.dumps(rejected, ensure_ascii=False),
        "corruption": corruption,
    })

print("Preference pairs:", len(preference_pairs))
print(json.dumps(preference_pairs[0], indent=2)[:4000])

## 14. Save the complete audit bundle

The export contains the LoRA adapter, tokenizer, configuration, training
history, train/validation/OOD case IDs, corpus provenance, preference pairs,
and benchmark outputs. The frozen base model is referenced by name rather than
copied into the archive.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(OUTPUT_DIR / "adapter")
tokenizer.save_pretrained(OUTPUT_DIR / "adapter")

manifest = {
    "created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "base_model": BASE_MODEL,
    "run_mode": RUN_MODE,
    "seed": SEED,
    "training_method": "4-bit NF4 QLoRA with prompt-masked causal LM loss",
    "config": {
        **CFG,
        "max_length": MAX_LENGTH,
        "micro_batch_size": MICRO_BATCH_SIZE,
        "gradient_accumulation_steps": GRAD_ACCUM_STEPS,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "lora_r": LORA_R,
        "lora_alpha": LORA_ALPHA,
        "lora_dropout": LORA_DROPOUT,
    },
    "counts": {
        "train_cases": len(train_cases),
        "validation_cases": len(val_cases),
        "ood_cases": len(ood_cases),
        "strict_ood_cases": len(strict_ood_cases),
        "qualitative_transfer_cases": len(qualitative_transfer_cases),
        "train_examples": len(train_rows),
        "validation_examples": len(val_rows),
        "optimizer_steps": optimizer_step,
    },
    "held_out_families": HOLDOUT_FAMILIES,
    "train_case_ids": sorted(train_ids),
    "validation_case_ids": sorted(val_ids),
    "ood_case_ids": sorted(ood_ids),
    "corpus_provenance": provenance,
    "limitations": [
        "Small-model behavior is not equivalent to human reasoning.",
        "Most supervision is procedural and template-generated.",
        "Verifier checks structure and evidence references, not ultimate truth.",
        "Keyword scoring is a weak proxy for conclusion correctness.",
        "Curated Holmes reconstructions require expert adjudication for research use.",
    ],
}

(OUTPUT_DIR / "manifest.json").write_text(
    json.dumps(manifest, indent=2), encoding="utf-8"
)
(OUTPUT_DIR / "training_history.json").write_text(
    json.dumps(history, indent=2), encoding="utf-8"
)
(OUTPUT_DIR / "benchmark.json").write_text(
    json.dumps(benchmark_rows, indent=2), encoding="utf-8"
)
(OUTPUT_DIR / "preference_pairs.jsonl").write_text(
    "\n".join(json.dumps(row, ensure_ascii=False) for row in preference_pairs),
    encoding="utf-8",
)

archive_path = Path("/content/holmesian_reasoning_bundle.zip")
with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for path in OUTPUT_DIR.rglob("*"):
        if path.is_file():
            zf.write(path, arcname=path.relative_to(OUTPUT_DIR.parent))

print("Saved:", archive_path)
print("Size (MB):", round(archive_path.stat().st_size / 1_000_000, 2))

In [ ]:
from google.colab import files
files.download("/content/holmesian_reasoning_bundle.zip")

## Interpretation and limits

If the tuned model improves JSON validity and evidence-reference precision, we
have shown that QLoRA can teach a **reasoning protocol**. We have not yet shown
that it discovers novel truths.

A research-grade continuation would add:

1. expert-annotated cases with inter-rater agreement;
2. adversarial cases where the obvious hypothesis is false;
3. process labels for each reasoning step;
4. a learned verifier trained independently from the policy model;
5. preference optimization using adjudicated good/bad trajectories;
6. domain-transfer tests in finance, operations, cybersecurity, and science;
7. calibration curves and selective abstention;
8. comparison against prompt-only and retrieval-augmented baselines.

The central result of this notebook is therefore methodological: **style data
teaches style; explicit, verifiable investigation episodes teach an
investigation procedure.**